# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"  Croissant ID: {metadata.id}\n  Version: {getattr(metadata, 'version', 'N/A')}\n  License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's print summary information about all record sets, their `@id`s, and the associated fields and columns.

In [ ]:
# List all record sets with their @id and schema details
record_sets = list(dataset.recordsets)
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"- Record set name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '')}")
    fields = getattr(rs, 'fields', [])
    print("  Fields (with @id):")
    for f in fields:
        print(f"    - {getattr(f, 'name', '')} (@id: {f.id}, type: {getattr(f, 'data_type', '')})")
    print("  Columns (with @id):")
    columns = getattr(rs, 'columns', [])
    for c in columns:
        print(f"    - {getattr(c, 'name', '')} (@id: {c.id})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll load all record sets and keep them in a dictionary of DataFrames by @id
dataframes = dict()
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        if records:  # Only if there is data
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            print(f"Loaded record set '{rs.name}' with @id '{rs.id}' as {len(df)} records, columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set '{rs.name}' (@id: {rs.id})")
    except Exception as ex:
        print(f"Error loading records for {rs.id}: {ex}")

# Pick the first available record set with data
main_record_set_id = next(iter(dataframes.keys()))
print(f"\nUsing '{main_record_set_id}' for downstream analysis.")
main_df = dataframes[main_record_set_id]
print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Reference fields and columns by their `@id`.**

In [ ]:
# Demonstrate EDA by referencing fields by @id
# We'll attempt to auto-detect a numeric field
numeric_candidate = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_candidate = col
        break

if numeric_candidate is None:
    # Try to coerce numeric columns if possible
    for col in main_df.columns:
        try:
            coerced = pd.to_numeric(main_df[col], errors='coerce')
            if coerced.notna().sum() > 0:
                main_df[col] = coerced
                numeric_candidate = col
                print(f"Coerced field '{col}' to numeric.")
                break
        except Exception:
            pass

print(f"Using numeric field for EDA: {numeric_candidate}")

# Set threshold for filtering
threshold = main_df[numeric_candidate].mean() if numeric_candidate else None
if numeric_candidate and threshold is not None:
    filtered_df = main_df[main_df[numeric_candidate] > threshold]
    print(f"Filtered records with {numeric_candidate} > {threshold:.2f} (mean):\n", filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_candidate}_normalized"] = (
        (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) /
        filtered_df[numeric_candidate].std()
    )
    print(f"\nNormalized {numeric_candidate} for filtered records:\n", filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"].copy()].head())

    # Pick a group-by field (try to find a string/categorical field)
    group_field = None
    for col in main_df.columns:
        if col != numeric_candidate and (main_df[col].dtype == object or pd.api.types.is_categorical_dtype(main_df[col])):
            group_field = col
            break

    print(f"\nUsing group field: {group_field}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().reset_index()
        print(f"\nGrouped data (mean {numeric_candidate}) by {group_field}:\n", grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field (referenced by its @id) and, if applicable, group means.

In [ ]:
import matplotlib.pyplot as plt

if numeric_candidate and numeric_candidate in main_df.columns:
    plt.figure(figsize=(8, 4))
    main_df[numeric_candidate].hist(bins=20, color='steelblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_candidate} (@id)")
    plt.xlabel(numeric_candidate)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df was created, plot group means
    if 'grouped_df' in locals() and group_field and not grouped_df.empty:
        plt.figure(figsize=(10, 4))
        grouped_df.plot(kind='bar', x=group_field, y=numeric_candidate, legend=False)
        plt.title(f"Mean {numeric_candidate} by {group_field}")
        plt.ylabel(f"Mean {numeric_candidate}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
In this notebook, we've demonstrated loading and inspecting the Mass Spectrometry Analysis of Extracellular Vesicles dataset using `mlcroissant`.

- The dataset contains high-dimensional proteomics data with rich metadata described via a Croissant schema.
- We accessed record sets, referenced all data fields and columns via their `@id`, and performed example filtering/grouping operations and visualization.

For robust downstream analysis, always consult the record set schema and field documentation using their `@id` for unambiguous data specification.